# Dice Fairness: Binomial Tests

Die faces: `['oo', 'o', 'xx', 'x', 'x', 'R']`

For each symbol, we run a one-vs-rest binomial test comparing observed count to expected probability from the die design.

In [ ]:
import sqlite3
from collections import Counter

import pandas as pd
from scipy.stats import binomtest

In [ ]:
DB_PATH = '../dice_rolls.db'  # adjust if needed
FACES = ['oo', 'o', 'xx', 'x', 'x', 'R']
expected_probs = {k: v / len(FACES) for k, v in Counter(FACES).items()}
expected_probs

In [ ]:
con = sqlite3.connect(DB_PATH)
rolls = pd.read_sql_query('SELECT result, rolled_at FROM rolls', con)
con.close()

rolls.head(), len(rolls)

In [ ]:
if rolls.empty:
    raise ValueError('No rolls found in database. Record some rolls first.')

n = len(rolls)
observed_counts = rolls['result'].value_counts().to_dict()
rows = []

for symbol, p_expected in sorted(expected_probs.items()):
    k = observed_counts.get(symbol, 0)
    test = binomtest(k=k, n=n, p=p_expected, alternative='two-sided')
    rows.append({
        'symbol': symbol,
        'observed_count': k,
        'total_rolls': n,
        'observed_p': k / n,
        'expected_p': p_expected,
        'p_value': test.pvalue,
    })

results = pd.DataFrame(rows).sort_values('p_value')
results

In [ ]:
alpha = 0.05
results['reject_at_0_05'] = results['p_value'] < alpha
results